# EarthForge — Spatial Analysis with GeoJSON Output

This notebook demonstrates a complete spatial analysis workflow that produces
GeoJSON outputs suitable for visualization in QGIS, ArcGIS, or web maps.

**Workflow:**
1. Search STAC for Sentinel-2 imagery over a study area
2. Build a GeoJSON FeatureCollection of scene footprints with cloud cover attributes
3. Fetch and convert KY Wildlife Management Areas to GeoParquet
4. Run spatial queries to identify protected areas within the study area
5. Export all results as GeoJSON files for downstream use

**Outputs:**
- `data/analysis/sentinel2_footprints.geojson` — scene footprints with cloud stats
- `data/analysis/wma_boundaries.geojson` — WMA polygons in the study area
- `data/analysis/coverage_grid.geojson` — data availability grid

Data sources: Element84 Earth Search (public), KY ArcGIS MapServer (public)

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../packages/core/src")
sys.path.insert(0, "../../packages/raster/src")
sys.path.insert(0, "../../packages/stac/src")
sys.path.insert(0, "../../packages/vector/src")

from earthforge.core.config import EarthForgeProfile

profile = EarthForgeProfile(
    name="earth-search",
    stac_api="https://earth-search.aws.element84.com/v1",
    storage_backend="local",
)

# Create output directory
OUTPUT_DIR = Path("data/analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 1. Search Sentinel-2 and Build Scene Footprints

Each STAC item has a bounding box and properties (cloud cover, platform, etc.).
We convert the search results into a GeoJSON FeatureCollection where each feature
is a scene footprint polygon with analysis-relevant attributes.

In [ ]:
from earthforge.stac.search import search_catalog

# Study area: Central Kentucky
STUDY_BBOX = [-85.0, 37.8, -84.2, 38.4]

results = await search_catalog(
    profile,
    collections=["sentinel-2-l2a"],
    bbox=STUDY_BBOX,
    datetime_range="2025-06-01/2025-09-30",
    max_items=30,
)

print(f"Found {len(results.items)} Sentinel-2 scenes")
print(f"Matched: {results.matched}")

In [ ]:
def bbox_to_polygon(bbox):
    """Convert [west, south, east, north] to a GeoJSON Polygon."""
    w, s, e, n = bbox
    return {
        "type": "Polygon",
        "coordinates": [[
            [w, s], [e, s], [e, n], [w, n], [w, s]
        ]]
    }


# Build GeoJSON FeatureCollection of scene footprints
features = []
for item in results.items:
    if not item.bbox:
        continue
    features.append({
        "type": "Feature",
        "geometry": bbox_to_polygon(item.bbox),
        "properties": {
            "id": item.id,
            "datetime": item.datetime,
            "collection": item.collection,
            "cloud_cover": item.properties.get("eo:cloud_cover"),
            "platform": item.properties.get("platform"),
            "asset_count": item.asset_count,
            "usable": (item.properties.get("eo:cloud_cover", 100) or 100) < 30,
        },
    })

footprints_geojson = {
    "type": "FeatureCollection",
    "features": features,
}

# Write to file
footprints_path = OUTPUT_DIR / "sentinel2_footprints.geojson"
footprints_path.write_text(json.dumps(footprints_geojson, indent=2))

usable = sum(1 for f in features if f["properties"]["usable"])
print(f"Wrote {len(features)} scene footprints to {footprints_path}")
print(f"Usable scenes (<30% cloud): {usable}/{len(features)}")
print(f"File size: {footprints_path.stat().st_size:,} bytes")

## 2. Fetch Wildlife Management Areas

Query the KY ArcGIS REST API for WMA polygons intersecting our study area,
then convert to GeoParquet and export as GeoJSON.

In [ ]:
import httpx
import tempfile

WMA_SERVICE = (
    "https://kygisserver.ky.gov/arcgis/rest/services/WGS84WM_Services"
    "/Ky_Public_Hunting_Areas_WGS84WM/MapServer/1"
)

# Spatial query: WMAs intersecting our study bbox
data = {
    "where": "1=1",
    "geometry": json.dumps({
        "xmin": STUDY_BBOX[0], "ymin": STUDY_BBOX[1],
        "xmax": STUDY_BBOX[2], "ymax": STUDY_BBOX[3],
        "spatialReference": {"wkid": 4326},
    }),
    "geometryType": "esriGeometryEnvelope",
    "spatialRel": "esriSpatialRelIntersects",
    "outFields": "AREANAME,WMA,ACRES_CAL,Counties,MANAGER,WILDMGTREG",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "json",
}

r = httpx.post(f"{WMA_SERVICE}/query", data=data, timeout=120)
r.raise_for_status()
esri_json = r.json()

# Convert Esri JSON to GeoJSON
wma_features = []
for feat in esri_json.get("features", []):
    attrs = feat.get("attributes", {})
    rings = feat.get("geometry", {}).get("rings", [])
    if not rings:
        continue
    wma_features.append({
        "type": "Feature",
        "properties": attrs,
        "geometry": {"type": "Polygon", "coordinates": rings},
    })

wma_geojson = {"type": "FeatureCollection", "features": wma_features}

# Write GeoJSON output
wma_path = OUTPUT_DIR / "wma_boundaries.geojson"
wma_path.write_text(json.dumps(wma_geojson, indent=2))

total_acres = sum(f["properties"].get("ACRES_CAL", 0) or 0 for f in wma_features)
print(f"Found {len(wma_features)} WMAs intersecting study area")
print(f"Total protected acres: {total_acres:,.0f}")
print(f"Wrote to {wma_path} ({wma_path.stat().st_size:,} bytes)")

## 3. Convert to GeoParquet and Spatial Query

Convert the WMA GeoJSON to GeoParquet using EarthForge's vector module,
then run a tighter spatial query to find WMAs in a specific sub-region.

In [ ]:
from earthforge.vector.convert import convert_vector
from earthforge.vector.query import query_features

# Convert GeoJSON -> GeoParquet
parquet_path = str(OUTPUT_DIR / "wma_boundaries.parquet")
result = await convert_vector(str(wma_path), output=parquet_path)

print(f"Converted {result.feature_count} features to GeoParquet")
print(f"  Format: {result.input_format} -> {result.output_format}")
print(f"  CRS:    {result.crs}")
print(f"  Size:   {result.file_size_bytes:,} bytes")

In [ ]:
# Spatial query: WMAs in the Lexington sub-region
lexington_bbox = [-84.6, 37.95, -84.3, 38.15]

query_result = await query_features(
    parquet_path,
    bbox=lexington_bbox,
    columns=["AREANAME", "ACRES_CAL", "Counties", "MANAGER"],
    include_geometry=False,
)

print(f"WMAs near Lexington: {query_result.feature_count}")
for f in query_result.features:
    name = f.get("AREANAME", "?")
    acres = f.get("ACRES_CAL", 0) or 0
    print(f"  {name:<30s} {acres:>8,.0f} acres")

## 4. Build a Data Availability Grid

Create a grid of cells over the study area, then for each cell count how many
usable Sentinel-2 scenes cover it. Export as GeoJSON for heatmap visualization.

This is a common pre-analysis step: understanding where you have the best
temporal coverage before committing to a time-series analysis.

In [ ]:
import statistics


def bbox_intersects(a, b):
    """Check if two [west, south, east, north] bboxes intersect."""
    return not (a[2] < b[0] or a[0] > b[2] or a[3] < b[1] or a[1] > b[3])


# Create a 4x4 grid over the study area
west, south, east, north = STUDY_BBOX
cols, rows = 4, 4
dx = (east - west) / cols
dy = (north - south) / rows

grid_features = []
for row in range(rows):
    for col in range(cols):
        cell_bbox = [
            west + col * dx,
            south + row * dy,
            west + (col + 1) * dx,
            south + (row + 1) * dy,
        ]

        # Count scenes and usable scenes covering this cell
        covering = []
        for item in results.items:
            if item.bbox and bbox_intersects(cell_bbox, item.bbox):
                covering.append(item)

        cloud_covers = [
            item.properties.get("eo:cloud_cover", 100)
            for item in covering
            if "eo:cloud_cover" in item.properties
        ]
        usable_count = sum(1 for cc in cloud_covers if (cc or 100) < 30)

        grid_features.append({
            "type": "Feature",
            "geometry": bbox_to_polygon(cell_bbox),
            "properties": {
                "row": row,
                "col": col,
                "total_scenes": len(covering),
                "usable_scenes": usable_count,
                "mean_cloud_cover": round(statistics.mean(cloud_covers), 1) if cloud_covers else None,
                "min_cloud_cover": round(min(cloud_covers), 1) if cloud_covers else None,
            },
        })

grid_geojson = {"type": "FeatureCollection", "features": grid_features}

grid_path = OUTPUT_DIR / "coverage_grid.geojson"
grid_path.write_text(json.dumps(grid_geojson, indent=2))

print(f"Wrote {len(grid_features)}-cell coverage grid to {grid_path}")
print()
print("Coverage heatmap (usable scenes per cell):")
for row in range(rows - 1, -1, -1):
    cells = [f for f in grid_features if f["properties"]["row"] == row]
    cells.sort(key=lambda c: c["properties"]["col"])
    vals = [str(c["properties"]["usable_scenes"]).rjust(3) for c in cells]
    print(f"  {'  '.join(vals)}")

## 5. Summary

All outputs are standard GeoJSON files that can be loaded directly into:
- **QGIS**: Layer > Add Layer > Add Vector Layer
- **ArcGIS Pro**: Map > Add Data > Browse to .geojson
- **Web maps**: Leaflet, MapLibre, Mapbox GL JS
- **Python**: `geopandas.read_file('data/analysis/sentinel2_footprints.geojson')`

The GeoParquet file (`wma_boundaries.parquet`) is the columnar equivalent
with predicate pushdown support for large-scale spatial queries.

In [ ]:
# List all outputs
print("Output files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {p.name:<40s} {p.stat().st_size:>10,} bytes")